[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan_boards.ipynb)

# 4chan API: boards

List every board with `boards.json` and read the per-board settings that matter for data collection.

The 4chan API is read-only JSON. It needs no account and no key. The only
dependency is `requests`, which Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
"Research considerations" section on the topic page before collecting.

In [1]:
import requests

In [2]:
api_url = "https://a.4cdn.org/boards.json"
resp = requests.get(api_url)
resp.status_code

200

In [3]:
resp_json = resp.json()
boards = resp_json["boards"]
len(boards)

77

In [4]:
boards[0]

{'board': '3',
 'title': '3DCG',
 'ws_board': 1,
 'per_page': 15,
 'pages': 10,
 'max_filesize': 4194304,
 'max_webm_filesize': 4194304,
 'max_comment_chars': 2000,
 'max_webm_duration': 120,
 'bump_limit': 310,
 'image_limit': 150,
 'cooldowns': {'threads': 600, 'replies': 60, 'images': 60},
 'meta_description': "&quot;/3/ - 3DCG&quot; is 4chan's board for 3D modeling and imagery.",
 'is_archived': 1}

The fields you will use:

| Field | Meaning |
|---|---|
| `board` | Short name, used in every other endpoint URL (`/g/`, `/pol/`) |
| `title` | Human-readable name |
| `ws_board` | 1 if the board is work-safe, 0 if not |
| `pages`, `per_page` | How many index pages the board has and how many threads each page holds. Their product is the number of live threads |
| `bump_limit` | After this many replies, new replies stop moving the thread to the top |
| `image_limit` | After this many images, new image posts are refused |
| `is_archived` | 1 if the board keeps an archive of expired threads. The key is absent, not 0, on boards without one, so read it with `.get()`. Boards without an archive cannot be collected through `archive.json` |
| `cooldowns` | Posting cooldowns in seconds. Not relevant for reading |

In [5]:
for board in boards[:10]:
    print(f"/{board['board']}/  {board['title']}  worksafe={board['ws_board']}  archive={board.get('is_archived', 0)}")

/3/  3DCG  worksafe=1  archive=1
/a/  Anime & Manga  worksafe=1  archive=1
/aco/  Adult Cartoons  worksafe=0  archive=1
/adv/  Advice  worksafe=1  archive=1
/an/  Animals & Nature  worksafe=1  archive=1
/b/  Random  worksafe=0  archive=0
/bant/  International/Random  worksafe=0  archive=0
/biz/  Business & Finance  worksafe=1  archive=1
/c/  Anime/Cute  worksafe=1  archive=1
/cgl/  Cosplay & EGL  worksafe=1  archive=1


In [6]:
# Boards without an archive
[board["board"] for board in boards if not board.get("is_archived")]

['b', 'bant', 'f', 'trash']

In [7]:
# Work-safe boards versus the rest
sum(board["ws_board"] for board in boards), len(boards)

(53, 77)